# 3D-ST · Generador gratuito de imagen a 3D

Este cuaderno ejecuta TripoSR en Google Colab y permite descargar la malla generada en formato **GLB** y **OBJ**.

**Uso:** ejecuta las celdas en orden. La instalación se hace una sola vez por sesión; después selecciona la imagen cuando se solicite. Colab puede no ofrecer GPU en todas las sesiones.

## 1. Comprobar GPU
En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** si está disponible. Luego ejecuta la celda.

In [ ]:
import subprocess, sys
print("Python del cuaderno:", sys.version)
try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception:
    print("No se detectó una GPU NVIDIA. En Colab, activa una GPU si hay disponibilidad y vuelve a ejecutar.")

## 2. Instalar el motor y sus dependencias
Esta celda prepara Python 3.11, PyTorch con CUDA 13.0, torchmcubes y TripoSR. Puede tardar varios minutos.

In [ ]:
import os, subprocess, textwrap, pathlib

def ejecutar(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

# Colab suele ofrecer Python 3.11 en este entorno; comprobar antes de instalar.
py311 = "/usr/bin/python3.11"
if not pathlib.Path(py311).exists():
    raise RuntimeError("Esta sesión no tiene Python 3.11 disponible. Reinicia el entorno de Colab y vuelve a intentarlo.")

if not pathlib.Path("/content/triposr-env/bin/python").exists():
    ejecutar([py311, "-m", "venv", "/content/triposr-env"])
PY = "/content/triposr-env/bin/python"
ejecutar([PY, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
ejecutar([PY, "-m", "pip", "install", "torch==2.11.0", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu130"])
ejecutar([PY, "-m", "pip", "install", "--upgrade", "scikit-build-core", "pybind11", "ninja", "cmake"])
ejecutar([PY, "-m", "pip", "install", "--no-cache-dir", "--no-build-isolation", "git+https://github.com/tatsy/torchmcubes.git"])
ejecutar([PY, "-m", "pip", "install", "omegaconf==2.3.0", "Pillow==10.1.0", "einops==0.7.0", "transformers==4.35.0", "trimesh==4.0.5", "rembg", "onnxruntime", "huggingface-hub", "imageio[ffmpeg]", "gradio", "xatlas==0.0.9", "moderngl==5.10.0"])
if not pathlib.Path("/content/TripoSR/run.py").exists():
    ejecutar(["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git", "/content/TripoSR"])

# Verificar imports y CUDA antes de intentar generar.
ejecutar([PY, "-c", "import torch, torchmcubes, rembg, onnxruntime; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('GPU disponible:', torch.cuda.is_available()); print('torchmcubes: OK')"])
print("Instalación completada.")

## 3. Subir la imagen
Selecciona una imagen JPG, PNG o WebP. Se recomienda que el objeto esté completo, centrado, bien iluminado y con un fondo sencillo.

In [ ]:
from google.colab import files
from pathlib import Path

subidos = files.upload()
if not subidos:
    raise RuntimeError("No se seleccionó ninguna imagen.")
nombre_original = next(iter(subidos.keys()))
imagen = str(Path("/content") / nombre_original)
print("Imagen seleccionada:", imagen)

## 4. Generar la pieza 3D
Esta celda produce los archivos GLB y OBJ. La primera ejecución puede descargar los pesos de TripoSR y del eliminador de fondo.

In [ ]:
import subprocess, pathlib, os
PY = "/content/triposr-env/bin/python"
TRIPOSR = "/content/TripoSR/run.py"
SALIDA = "/content/3dst-output"
os.makedirs(SALIDA, exist_ok=True)

comando = [PY, TRIPOSR, imagen, "--output-dir", SALIDA, "--model-save-format", "glb"]
print("Iniciando generación. Puede tardar varios minutos...")
resultado = subprocess.run(comando, text=True)
if resultado.returncode != 0:
    raise RuntimeError("TripoSR no pudo generar la malla. Revisa los mensajes de error anteriores.")

archivos = [p for p in pathlib.Path(SALIDA).rglob("*") if p.is_file()]
if not archivos:
    raise RuntimeError("TripoSR terminó, pero no se encontraron archivos de salida.")
print("Archivos generados:")
for p in archivos:
    print(f"{p} — {p.stat().st_size / 1024:.1f} KB")

## 5. Descargar el modelo
Descarga el archivo GLB para visualizarlo en aplicaciones compatibles. Si se generó un OBJ, también se ofrecerá para descargar.

In [ ]:
from google.colab import files
from pathlib import Path

salida = Path("/content/3dst-output")
modelos = list(salida.rglob("*.glb")) + list(salida.rglob("*.obj"))
if not modelos:
    raise FileNotFoundError("No se encontró ningún archivo GLB u OBJ en la carpeta de salida.")
for modelo in modelos:
    print("Descargando:", modelo.name)
    files.download(str(modelo))

## Importante
TripoSR estima la geometría a partir de una sola imagen; no garantiza dimensiones exactas ni que la pieza esté lista para impresión 3D. Puede requerir reparación de malla y comprobación en un laminador. La GPU gratuita de Colab depende de disponibilidad y la sesión puede terminarse.